[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarinamansor/ACE6313_T2610/blob/main/Ch2_2_DataSplitting.ipynb)

#**Chapter 2: Data Splitting, Performance Metric, Hyperparameter Tuning**

##1. Data Splitting

In [ ]:
#Load Dataset
from pandas import read_csv
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
df = read_csv('https://raw.githubusercontent.com/sarinamansor/ML_Datasets/main/pima-indians-diabetes.data.csv', names=names)

array = df.values
X = array[:, :-1]
y = array[:, -1]

In [ ]:
print(X.shape)

In [ ]:
#Split the dataset to training (67%), test (33%).
from sklearn.model_selection import train_test_split as split
X_train, X_test, y_train, y_test = split(X, y, test_size=0.33, random_state=42)

In [ ]:
print(f'X_train = {X_train.shape}')
print(f'X_test = {X_test.shape}')
print(f'y_train = {y_train.shape}')
print(f'y_test = {y_test.shape}')

###Hold-out Validation

In [ ]:
#Evaluate using a train and a test set
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier() #creating the model
model.fit(X_train, y_train) #train the model
result = model.score(X_test, y_test)
print(f'Accuracy: {result:.2%}')

###K-fold Cross-validation

In [ ]:
#Evaluate using Cross Validation
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier() #create model
kfold = KFold(n_splits=5, shuffle=True, random_state=42) #define cross-validation parameters
results = cross_val_score(model, X, y, cv=kfold) #perform cross-validation kfold times

print(f'results = {results}')
print(f'Accuracy: {results.mean():.2%} ({results.std():.2%})')

##2. Performance Metrics

###Confusion Matrix

In [ ]:
#Classification Confusion Matrix

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

X_train, X_test, y_train, y_test = split(X, y, test_size=0.33, random_state=42)
model = KNeighborsClassifier()
model.fit(X_train, y_train)
cf = confusion_matrix(y_test, model.predict(X_test))
ConfusionMatrixDisplay(cf, display_labels=['0','1']).plot()

In [ ]:
#generate classification report
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
#print(classification_report(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Normal (Negative)', 'Infected (Positive)']))

###Generate Visual Classification Report using Yellowbrick

In [ ]:
#Classification report using Yellowbrick
from yellowbrick.classifier import ClassificationReport

X_train, X_test, y_train, y_test = split(X, y, test_size=0.33, random_state=42)
model = KNeighborsClassifier()
model.fit(X_train, y_train)
report = ClassificationReport(model, classes = ['Normal (Negative)', 'Infected (Positive)'])
report.fit(X_train, y_train)
report.score(X_test, y_test)
report.show()

##3. Hyperparameter Tuning

In [ ]:
#Hyperparameter tuning with grid search
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split as split, KFold

X_train, X_test, y_train, y_test = split(X, y, random_state=42)
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
print(f'Accuracy without tuning: {model.score(X_test, y_test):.2%}')

#Grid Search + Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
params = dict(criterion=['gini','entropy'], max_leaf_nodes=range(2,21))
grid = GridSearchCV(model, params, cv=kf, n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

#Get the best parameters and retrain model with best parameters
print(grid.best_params_)
model.set_params(**grid.best_params_)
model.fit(X_train, y_train)

print(f'Accuracy with tuning: {model.score(X_test, y_test):.2%}')